# 売上予測ハンズオン

## 1. 事前準備

分析に必要なパッケージをインストール・インポートします。

In [ ]:
# パッケージをインストール
%pip install statsmodels matplotlib pandas

In [ ]:
# パッケージのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
# テーブルから読み込み＆データフレーム化
fact = spark.read.table("fact_sales_v2").toPandas()

# 2. Data Wrangler によるデータ加工

> **Data Wrangler とは?**  
> ノーコードでデータ加工ができるツールです。  
> GUI による直感的な操作で、自動的に Python コードが生成されます。

売上予測を行うために、fact データを月次売上データに集計します。
必要なデータ変換は以下の通り。

1. 分析に不要な列を削除
2. 日次データを月次に集計（各月の売上合計）
3. 列名を変更

In [ ]:
# 加工前のデータをディスプレイ関数で確認
display(fact)

In [ ]:
# # Pandas DataFrame の Data Wrangler によって生成されたコード

# def clean_data(fact):
#     # Keep only date and sales_amount columns
#     fact = fact[['date', 'sales_amount']]
#     # Set all dates to first of month
#     fact['date'] = pd.to_datetime(fact['date']).dt.to_period('M').dt.to_timestamp()
#     # 列でグループ化された 1 件の集計 が実行されました: 'date'
#     fact = fact.groupby(['date']).agg(sales_amount_sum=('sales_amount', 'sum')).reset_index()
#     # 列名を'date'から'month_start'に変更
#     fact = fact.rename(columns={'date': 'month_start'})
#     # 列名を'sales_amount_sum'から'monthly_sales'に変更
#     fact = fact.rename(columns={'sales_amount_sum': 'monthly_sales'})
#     return fact

# fact_clean = clean_data(fact.copy())
# display(fact_clean)

In [ ]:
# # 加工済みデータを Lakehouse Files へ保存
# output_path = "/lakehouse/default/Files/csv/monthly_sales.csv"

# fact_clean.to_csv(output_path, index=False)
# print(f"Saved to Lakehouse Files: {output_path}")

# 3. 売上予測

時系列予測とは、過去のデータのパターンを分析し、将来の値を予測する手法です。

**主なユースケース**
- 売上予測
- 中長期の事業計画・予算策定
- 在庫の受発注
- 人員配置・シフト計画の最適化
- プロモーション施策の実施タイミング判断

## 3-1. データの観察

まず、データを可視化して傾向を把握しましょう。

**なぜ可視化が重要？**
- トレンド（上昇・下降傾向）があるか？
- 季節性（毎年同じパターン）があるか？
- 異常値や欠損はないか？

これらを確認することで、予測可能性および適切なモデルを選択できます。

In [ ]:
# 月次売上の可視化
plt.figure(figsize=(12, 5))
plt.plot(fact_clean["month_start"], fact_clean["monthly_sales"], marker='o')
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales (JPY)")
plt.grid(True, alpha=0.3)
plt.show()

## 3-2. Train / Test 分割
モデルの性能を正しく評価するため、データを**訓練用**と**テスト用**に分割します。

| データ | 期間 | 用途 |
|--------|------|------|
| **Train（訓練）** | 2022/01 〜 2024/12 | モデルの学習に使用 |
| **Test（テスト）** | 2025/01 〜 2025/12 | 予測精度の評価に使用 |

> **重要**: テストデータは訓練時に使用しません。これにより「未知のデータ」に対する予測性能を評価できます。

In [ ]:
# 2025年分データをテストデータとして分割
train_data = fact_clean[fact_clean["month_start"].dt.year < 2025].copy()
test_data = fact_clean[fact_clean["month_start"].dt.year == 2025].copy()

print(f"Train: {len(train_data)} months ({train_data['month_start'].min()} ~ {train_data['month_start'].max()})")
print(f"Test:  {len(test_data)} months ({test_data['month_start'].min()} ~ {test_data['month_start'].max()})")

## 3-3. モデルの構築

SARIMA（Seasonal ARIMA） は、過去のデータを学習して将来を予測する統計ベースの時系列モデルです。

以下の3つの要素を同時にモデル化します。

- **トレンド**: 長期的な上昇・下降傾向
- **季節性**: 一定周期で繰り返すパターン
- **自己相関**: 過去の値と現在の値の関係性




パラメータ構成:

```
SARIMA(p, d, q)(P, D, Q, s)
       └─────┘ └────────┘
        非季節性   季節性
```

| パラメータ | 役割 | 今回の設定 |
|-----------|------|-----------|
| `p, d, q` | 短期パターンの捕捉 | (1, 1, 1) <br>※ 直近1か月の動きと増減を考慮|
| `P, D, Q` | 季節パターンの捕捉 | (1, 1, 1) <br>※ 去年の同月の影響を考慮|
| `s` | 季節周期 | 12 <br>※ 月次|

In [ ]:
# 時系列インデックスを設定
train_series = train_data.set_index("month_start")["monthly_sales"]
train_series.index = pd.DatetimeIndex(train_series.index).to_period('M').to_timestamp()
train_series = train_series.asfreq('MS')  # Month Start frequency

# SARIMA モデル構築
# order=(p, d, q), seasonal_order=(P, D, Q, s)
# 月次データなので季節周期 s=12
model = SARIMAX(
    train_series,
    order=(1, 1, 1),               # (p, d, q)
    seasonal_order=(1, 1, 1, 12),  # (P, D, Q, s)
    enforce_stationarity=False,
    enforce_invertibility=False
)

# モデルのフィッティング
print("SARIMA モデルをフィッティング中...")
result = model.fit(disp=False)
print("\nモデルサマリ:")
print(result.summary())

## 3-4. 予測と評価

構築したモデルで 2025年の売上を予測し、実際の値と比較します。

In [ ]:
# 2025年（テスト期間）を予測
n_forecast = len(test_data)
forecast = result.get_forecast(steps=n_forecast)
y_pred = forecast.predicted_mean.values
y_test = test_data["monthly_sales"].values

# 予測結果の確認
pred_df = pd.DataFrame({
    "month": test_data["month_start"].values,
    "actual": y_test,
    "predicted": y_pred,
    "error": y_test - y_pred,
    "error_pct": (y_test - y_pred) / y_test * 100
})
display(pred_df)

予測の精度を測定するために、以下の指標を使用します：

| 指標 | 説明 | 解釈 |
|------|------|------|
| **MAPE** | 平均絶対パーセント誤差 | 「平均して何%ズレているか」 |
| **MAE** | 平均絶対誤差 | 「平均していくらズレているか」（円単位） |
| **RMSE** | 二乗平均平方根誤差 | 大きな誤差をより重視した指標 |


In [ ]:
# MAPE の計算
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(1e-9, np.abs(y_true)))) * 100

def mae(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))

def rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

print("=" * 50)
print("SARIMA モデル評価結果 (2025年)")
print("=" * 50)
print(f"MAPE: {mape(y_test, y_pred):.2f}%")
print(f"MAE:  {mae(y_test, y_pred):,.0f} JPY")
print(f"RMSE: {rmse(y_test, y_pred):,.0f} JPY")
print("=" * 50)

## 3-5. 評価結果の可視化

予測結果をグラフで確認します。

- **上のグラフ**: 全期間の実績と予測（信頼区間付き）
- **下のグラフ**: 2025年の月別比較（棒グラフ）

In [ ]:
# 全期間 + 2025年予測の可視化
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Plot 1: 全期間の実績 vs 予測 ---
ax1 = axes[0]
ax1.plot(fact_clean["month_start"], fact_clean["monthly_sales"], 
         label="Actual", marker='o', linewidth=2)
ax1.plot(test_data["month_start"], y_pred, 
         label="Predicted (2025)", marker='s', linewidth=2, color='red')

# 信頼区間 (90%)
conf_int = forecast.conf_int(alpha=0.25)
ax1.fill_between(
    test_data["month_start"],
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    alpha=0.2, color='red', label='75% Confidence Interval'
)

ax1.axvline(x=pd.Timestamp('2025-01-01'), color='gray', linestyle='--', alpha=0.7, label='Train/Test Split')
ax1.set_title("Monthly Sales: Actual vs SARIMA Prediction", fontsize=14)
ax1.set_xlabel("Month")
ax1.set_ylabel("Sales (JPY)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: 2025年のみ拡大表示 ---
ax2 = axes[1]
months_2025 = test_data["month_start"]
width = 10  # bar width in days

x = np.arange(len(months_2025))
ax2.bar(x - 0.2, y_test, width=0.4, label='Actual', color='steelblue', alpha=0.8)
ax2.bar(x + 0.2, y_pred, width=0.4, label='Predicted', color='coral', alpha=0.8)

ax2.set_xticks(x)
ax2.set_xticklabels([d.strftime('%Y-%m') for d in months_2025], rotation=45)
ax2.set_title(f"2025年 月次売上: Actual vs Predicted (MAPE: {mape(y_test, y_pred):.2f}%)", fontsize=14)
ax2.set_xlabel("Month")
ax2.set_ylabel("Sales (JPY)")
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 4. ベースラインとの比較 (Seasonal Naive)

モデルの価値を評価するため、シンプルな手法（ベースライン）と比較します。

### Seasonal Naive とは？
「**前年同月の値をそのまま予測値とする**」最もシンプルな手法です。

> **なぜベースラインと比較する？**
> - 複雑なモデルが「シンプルな手法より良いか」を確認
> - ベースラインに負けるなら、モデルの改善が必要
> - 実務では「最低限これくらいの精度は出せる」という基準になる（≒本番導入への指標）

In [ ]:
# Seasonal Naive: 前年同月（2024年の値を使用）
prev_year_data = fact_clean[
    (fact_clean["month_start"].dt.year == 2024) & 
    (fact_clean["month_start"].dt.month.isin(test_data["month_start"].dt.month))
].sort_values("month_start")

y_seasonal_naive = prev_year_data["monthly_sales"].values[:len(y_test)]

print("=" * 60)
print("モデル比較")
print("=" * 60)
print(f"SARIMA        - MAPE: {mape(y_test, y_pred):6.2f}%")
print(f"Seasonal Naive - MAPE: {mape(y_test, y_seasonal_naive):6.2f}%")
print("=" * 60)

if mape(y_test, y_pred) < mape(y_test, y_seasonal_naive):
    print("→ SARIMA モデルがベースラインを上回っています！")
else:
    print("→ Seasonal Naive の方が良い結果です。モデルの改善が必要かもしれません。")